In [1]:
!pip install -q datasets transformers huggingface_hub safetensors -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 143.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.6 MB/s eta 0:00:00


In [2]:
import os
from datasets import load_dataset
from transformers import AutoTokenizer

# Clean any stale/partial .bin files before regenerating
for f in ["train.bin", "test.bin"]:
    if os.path.exists(f):
        os.remove(f)

ds = load_dataset("Ananda100/python-cleanss", split="train")
ds = ds.train_test_split(test_size=0.01, seed=42)

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base", trust_remote_code=True)
vocab_size = len(tokenizer)
print(f"Tokenizer vocab size: {vocab_size}")

if tokenizer.eos_token_id is None:
    raise ValueError("Tokenizer has no eos_token_id set.")

README.md:   0%|          | 0.00/289 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

data/train-00000-of-00019.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/train-00000-of-00019.parquet: downloading bytes:           |  0.00B            

...

Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/793 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

Tokenizer vocab size: 32022


In [3]:
import numpy as np
from tqdm.auto import tqdm

def process(example):
    col = 'code' if 'code' in example else 'text' if 'text' in example else 'content'
    ids = tokenizer.encode(example[col], add_special_tokens=False) + [tokenizer.eos_token_id]
    return {'ids': ids, 'len': len(ids)}

tokenized = ds.map(
    process,
    remove_columns=ds['train'].column_names,
    desc="tokenizing the splits",
    num_proc=8,
)

for split, dset in tokenized.items():
    arr_len = np.sum(dset['len'], dtype=np.uint64)
    filename = f'{split}.bin'
    arr = np.memmap(filename, dtype=np.uint16, mode='w+', shape=(arr_len,))
    total_batches = 1024
    idx = 0
    for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):
        batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')
        arr_batch = np.concatenate(batch['ids'])
        arr[idx: idx + len(arr_batch)] = arr_batch
        idx += len(arr_batch)
    arr.flush()

train_data = np.memmap("train.bin", dtype=np.uint16, mode="r")
val_data = np.memmap("test.bin", dtype=np.uint16, mode="r")
print(f"Total training tokens: {len(train_data):,}")
print(f"Total validation tokens: {len(val_data):,}")

tokenizing the splits (num_proc=8):   0%|          | 0/1782000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (18530 > 16384). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (16432 > 16384). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (17229 > 16384). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (17430 > 16384). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (16629 > 16384). Running this sequence through the model will result in indexing er

tokenizing the splits (num_proc=8):   0%|          | 0/18000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (16818 > 16384). Running this sequence through the model will result in indexing errors


writing train.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

writing test.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

Total training tokens: 2,955,981,351
Total validation tokens: 30,265,069


In [4]:
from huggingface_hub import HfApi

HF_TOKEN = ""  # <-- fill in
TOKENIZED_DATASET_REPO = "Ananda100/python-cleanss-tokenized"

if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=TOKENIZED_DATASET_REPO, repo_type="dataset", exist_ok=True, token=HF_TOKEN)
    api.upload_file(path_or_fileobj="train.bin", path_in_repo="train.bin",
                     repo_id=TOKENIZED_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
    api.upload_file(path_or_fileobj="test.bin", path_in_repo="test.bin",
                     repo_id=TOKENIZED_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
    print(f"Uploaded .bin files to https://huggingface.co/datasets/{TOKENIZED_DATASET_REPO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/train.bin          :   0%|          |  642kB / 5.91GB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/test.bin           :   1%|1         |  647kB / 60.5MB            

Uploaded .bin files to https://huggingface.co/datasets/Ananda100/python-cleanss-tokenized


In [24]:
import torch
from contextlib import nullcontext

BASE_STUDENT_REPO = "Ananda100/pocketcoder100M"
TEACHER_MODEL_ID  = "deepseek-ai/deepseek-coder-6.7b-base"
DISTILLED_REPO_ID = "Ananda100/pocketcoder100M-distilled"

TEMPERATURE  = 2.0
ALPHA        = 0.7
DISTILL_LR   = 5e-5
WARMUP_STEPS = 200
MIN_LR       = 5e-6

batch_size = 8       # reduce further (e.g. 4) if you OOM with teacher loaded
block_size = 512
gradient_accumulation_steps = 32
tokens_per_step = batch_size * block_size * gradient_accumulation_steps

DISTILL_EPOCH_FRACTION = 0.10   # refinement pass, not a full epoch
eval_iters = 50

device = "cuda" if torch.cuda.is_available() else "cpu"
device_type = "cuda" if "cuda" in device else "cpu"
dtype = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float16"
ptdtype = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[dtype]
ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast(device_type=device_type, dtype=ptdtype)
torch.manual_seed(42)

print(f"Targeting distillation epoch fraction: {DISTILL_EPOCH_FRACTION}")

Targeting distillation epoch fraction: 0.1


In [25]:
def get_batch(split):
    data = np.memmap("train.bin" if split == "train" else "test.bin", dtype=np.uint16, mode="r")
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    if device_type == 'cuda':
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

In [26]:
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass, asdict

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
                dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 32256
    n_layer: int = 10
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1
    bias: bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        dev = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=dev)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, eos_token_id=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
            if eos_token_id is not None and idx.size(0) == 1 and idx_next.item() == eos_token_id:
                break
        return idx

In [27]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import json

config_path = hf_hub_download(repo_id=BASE_STUDENT_REPO, filename="config.json", token=HF_TOKEN or None)
weights_path = hf_hub_download(repo_id=BASE_STUDENT_REPO, filename="model.safetensors", token=HF_TOKEN or None)

with open(config_path) as f:
    student_config_dict = json.load(f)

student_config = GPTConfig(**student_config_dict)
student_config.vocab_size = vocab_size

student = GPT(student_config)
student.load_state_dict(load_file(weights_path))
student.to(device)
student.train()
print(f"Student loaded. Params: {sum(p.numel() for p in student.parameters())/1e6:.2f} M")

Student loaded. Params: 95.87 M


In [28]:
from transformers import AutoModelForCausalLM

teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID, trust_remote_code=True,
    torch_dtype=torch.bfloat16 if device_type == "cuda" else torch.float32,
)
teacher.to(device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False
print(f"Teacher output vocab dim: {teacher.get_output_embeddings().weight.shape[0]}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Teacher output vocab dim: 32256


In [35]:
def distillation_loss(student_logits, teacher_logits, targets, temperature=TEMPERATURE, alpha=ALPHA):
    v = min(student_logits.size(-1), teacher_logits.size(-1))
    student_logits = student_logits[..., :v]
    teacher_logits = teacher_logits[..., :v]
    student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)
    teacher_probs = F.softmax(teacher_logits / temperature, dim=-1)
    kd_loss = F.kl_div(student_log_probs.reshape(-1, v), teacher_probs.reshape(-1, v),
                        reduction="batchmean") * (temperature ** 2)
    ce_loss = F.cross_entropy(student_logits.reshape(-1, v), targets.reshape(-1), ignore_index=-1)
    total_loss = alpha * kd_loss + (1 - alpha) * ce_loss
    return total_loss, kd_loss, ce_loss

In [36]:
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

train_data = np.memmap("train.bin", dtype=np.uint16, mode="r")
target_tokens = int(len(train_data) * DISTILL_EPOCH_FRACTION)
max_iters = max(1, int(target_tokens / tokens_per_step))
total_loop_iters = max_iters * gradient_accumulation_steps
print(f"Distillation optimizer steps: {max_iters:,} (micro-steps: {total_loop_iters:,})")

optimizer = torch.optim.AdamW(student.parameters(), lr=DISTILL_LR, betas=(0.9, 0.95), weight_decay=0.1, eps=1e-9)
scheduler_warmup = LinearLR(optimizer, total_iters=WARMUP_STEPS)
scheduler_decay = CosineAnnealingLR(optimizer, T_max=max(1, max_iters - WARMUP_STEPS), eta_min=MIN_LR)
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[WARMUP_STEPS])
scaler = torch.amp.GradScaler(device_type, enabled=(dtype == 'float16'))

Distillation optimizer steps: 2,255 (micro-steps: 72,160)


In [37]:
@torch.no_grad()
def estimate_loss():
    student.eval()
    out = {}
    for split in ['train', 'val']:
        total_losses = torch.zeros(eval_iters)
        kd_losses = torch.zeros(eval_iters)
        ce_losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with ctx:
                student_logits, _ = student(X, Y)
                teacher_logits = teacher(X).logits
                total_loss, kd_loss, ce_loss = distillation_loss(student_logits, teacher_logits, Y)
            total_losses[k] = total_loss.item()
            kd_losses[k] = kd_loss.item()
            ce_losses[k] = ce_loss.item()
        out[split] = {
            'total': total_losses.mean(),
            'kd': kd_losses.mean(),
            'ce': ce_losses.mean(),
        }
    student.train()
    return out

In [38]:
best_val_loss = float("inf")
distilled_ckpt_path = "distilled_coder_model_100m.pt"

for step in tqdm(range(total_loop_iters)):
    if step % (eval_iters * gradient_accumulation_steps) == 0 and step != 0:
        losses = estimate_loss()
        optimizer_step = step // gradient_accumulation_steps
        print(
            f"Step {optimizer_step}: "
            f"train [total={losses['train']['total']:.4f}, kd={losses['train']['kd']:.4f}, ce={losses['train']['ce']:.4f}] | "
            f"val [total={losses['val']['total']:.4f}, kd={losses['val']['kd']:.4f}, ce={losses['val']['ce']:.4f}]"
        )
        if losses['val']['total'] < best_val_loss:
            best_val_loss = losses['val']['total']
            torch.save(student.state_dict(), distilled_ckpt_path)

    X, Y = get_batch("train")
    with ctx:
        student_logits, _ = student(X, Y)
        with torch.no_grad():
            teacher_logits = teacher(X).logits
        total_loss, kd_loss, ce_loss = distillation_loss(student_logits, teacher_logits, Y)
        loss = total_loss / gradient_accumulation_steps
        scaler.scale(loss).backward()

    if ((step + 1) % gradient_accumulation_steps == 0) or (step + 1 == total_loop_iters):
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=0.5)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

    # optional: print every micro-step's raw KD/CE too
    if step % 50 == 0:
        print(f"  micro-step {step}: kd={kd_loss.item():.4f}, ce={ce_loss.item():.4f}, total={total_loss.item():.4f}")

print("--- Distillation complete ---")
torch.save(student.state_dict(), distilled_ckpt_path)

  0%|          | 0/72160 [00:00<?, ?it/s]

  micro-step 0: kd=2.1194, ce=1.5041, total=1.9348
  micro-step 50: kd=2.3515, ce=1.5584, total=2.1135
  micro-step 100: kd=2.1472, ce=1.6583, total=2.0005
  micro-step 150: kd=1.8688, ce=1.3212, total=1.7045
  micro-step 200: kd=3.2266, ce=1.8874, total=2.8248
  micro-step 250: kd=1.9995, ce=1.8325, total=1.9494
  micro-step 300: kd=1.8378, ce=1.5902, total=1.7636
  micro-step 350: kd=1.6991, ce=1.6408, total=1.6816
  micro-step 400: kd=2.0134, ce=1.5537, total=1.8755
  micro-step 450: kd=1.8686, ce=1.4777, total=1.7513
  micro-step 500: kd=1.6642, ce=1.2925, total=1.5527
  micro-step 550: kd=2.2896, ce=1.3763, total=2.0156
  micro-step 600: kd=1.9116, ce=1.5548, total=1.8046
  micro-step 650: kd=2.2113, ce=1.3871, total=1.9640
  micro-step 700: kd=1.9767, ce=1.7768, total=1.9168
  micro-step 750: kd=1.9649, ce=1.6044, total=1.8567
  micro-step 800: kd=2.0829, ce=1.7107, total=1.9712
  micro-step 850: kd=1.8785, ce=1.3825, total=1.7297
  micro-step 900: kd=1.8817, ce=1.6275, total=1.8

In [41]:
from huggingface_hub import HfApi, create_repo

student.load_state_dict(torch.load(distilled_ckpt_path, map_location=device))

upload_dir = "hf_upload_distilled"
os.makedirs(upload_dir, exist_ok=True)

state_dict = {k: v.contiguous().clone() for k, v in student.state_dict().items()}
from safetensors.torch import save_file
save_file(state_dict, os.path.join(upload_dir, "model.safetensors"))

with open(os.path.join(upload_dir, "config.json"), "w") as f:
    json.dump(asdict(student_config), f, indent=2)

tokenizer.save_pretrained(upload_dir)

readme = f"""---
license: apache-2.0
tags:
  - nanoGPT
  - code-generation
  - knowledge-distillation
---

# {DISTILLED_REPO_ID.split('/')[-1]}

A ~100M parameter GPT-style (nanoGPT architecture) language model, pretrained
from scratch on Python code, then refined with logit-based knowledge
distillation from `{TEACHER_MODEL_ID}`.

- **Base model (pretrained only):** [{BASE_STUDENT_REPO}](https://huggingface.co/{BASE_STUDENT_REPO})
- **Teacher model:** `{TEACHER_MODEL_ID}`
- **Distillation:** logit-based KD (Hinton et al.), temperature={TEMPERATURE}, alpha={ALPHA}
- Block size: {student_config.block_size} | Vocab size: {student_config.vocab_size}
"""
with open(os.path.join(upload_dir, "README.md"), "w") as f:
    f.write(readme)

api = HfApi(token=HF_TOKEN)
create_repo(repo_id=DISTILLED_REPO_ID, token=HF_TOKEN, exist_ok=True)
api.upload_folder(folder_path=upload_dir, repo_id=DISTILLED_REPO_ID, token=HF_TOKEN)
print(f"Uploaded to https://huggingface.co/{DISTILLED_REPO_ID}")

No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded to https://huggingface.co/Ananda100/pocketcoder100M-distilled


In [58]:
import torch

# Load base (pretrained-only) student
base_config = GPTConfig(**student_config_dict)
base_config.vocab_size = vocab_size
base_model = GPT(base_config)
base_model.load_state_dict(load_file(weights_path))  # original pretrained weights
base_model.to(device).eval()

# Distilled model is already `student` after training, or reload from checkpoint
distilled_model = GPT(student_config)
distilled_model.load_state_dict(torch.load(distilled_ckpt_path, map_location=device))
distilled_model.to(device).eval()
test_prompts = [

# =========================
# IMPORTS (50)
# =========================
"import os\n",
"import sys\n",
"import re\n",
"import json\n",
"import time\n",
"import math\n",
"import random\n",
"import logging\n",
"import argparse\n",
"import pathlib\n",
"import shutil\n",
"import subprocess\n",
"import asyncio\n",
"import threading\n",
"import multiprocessing\n",
"import functools\n",
"import itertools\n",
"import collections\n",
"import heapq\n",
"import pickle\n",
"import sqlite3\n",
"import csv\n",
"import yaml\n",
"import requests\n",
"import numpy as np\n",
"import pandas as pd\n",
"import matplotlib.pyplot as plt\n",
"import scipy\n",
"import sklearn\n",
"import tensorflow as tf\n",
"import torch\n",
"import torchvision\n",
"import torch.nn as nn\n",
"import torch.optim as optim\n",
"from sklearn.model_selection import ",
"from sklearn.preprocessing import ",
"from sklearn.metrics import ",
"from sklearn.datasets import ",
"from collections import ",
"from datetime import ",
"from pathlib import ",
"from flask import ",
"from fastapi import ",
"from sqlalchemy import ",
"from django.db import ",
"from django.shortcuts import ",
"from django.urls import ",
"from contextlib import ",
"from dataclasses import ",
"from typing import ",

# =========================
# FUNCTIONS (50)
# =========================
"def train(model):\n",
"def evaluate(model):\n",
"def predict(model, x):\n",
"def load_data(path):\n",
"def save_model(model, path):\n",
"def preprocess(text):\n",
"def tokenize(text):\n",
"def collate_fn(batch):\n",
"def compute_loss(pred, target):\n",
"def accuracy(pred, target):\n",
"def precision(pred, target):\n",
"def recall(pred, target):\n",
"def f1_score(pred, target):\n",
"def softmax(x):\n",
"def sigmoid(x):\n",
"def relu(x):\n",
"def normalize(data):\n",
"def batch_generator(data):\n",
"def parse_json(file):\n",
"def read_csv(path):\n",
"def write_csv(path, df):\n",
"def download_file(url):\n",
"def send_request(url):\n",
"def fibonacci(n):\n",
"def binary_search(arr, target):\n",
"def merge_sort(arr):\n",
"def quicksort(arr):\n",
"def dfs(graph, start):\n",
"def bfs(graph, start):\n",
"def topological_sort(graph):\n",
"def shortest_path(graph):\n",
"def is_palindrome(s):\n",
"def reverse_string(s):\n",
"def count_words(text):\n",
"def remove_duplicates(lst):\n",
"def flatten(lst):\n",
"def moving_average(data):\n",
"def split_dataset(data):\n",
"def seed_everything(seed):\n",
"def set_random_seed(seed):\n",
"def save_checkpoint(model):\n",
"def load_checkpoint(path):\n",
"def parse_args():\n",
"def build_model():\n",
"def train_epoch(model):\n",
"def validate(model):\n",
"def inference(model):\n",
"def compute_metrics(preds):\n",
"def create_dataset(path):\n",
"def main():\n",

# =========================
# CLASSES (40)
# =========================
"class Dataset:\n",
"class Trainer:\n",
"class Model:\n",
"class Config:\n",
"class CNN(nn.Module):\n",
"class MLP(nn.Module):\n",
"class Transformer(nn.Module):\n",
"class LSTM(nn.Module):\n",
"class GRU(nn.Module):\n",
"class AutoEncoder(nn.Module):\n",
"class ResNet(nn.Module):\n",
"class UNet(nn.Module):\n",
"class BinaryTree:\n",
"class LinkedList:\n",
"class Graph:\n",
"class Node:\n",
"class Queue:\n",
"class Stack:\n",
"class Solution:\n",
"class Logger:\n",
"class DataLoader:\n",
"class Tokenizer:\n",
"class Vocabulary:\n",
"class Optimizer:\n",
"class Scheduler:\n",
"class Metrics:\n",
"class EarlyStopping:\n",
"class Checkpoint:\n",
"class FlaskApp:\n",
"class FastAPIApp:\n",
"class User(models.Model):\n",
"class Product(models.Model):\n",
"class TestModel(unittest.TestCase):\n",
"class MyException(Exception):\n",
"class ConfigParser:\n",
"class Database:\n",
"class SQLModel:\n",
"class APIClient:\n",
"class ImageDataset(Dataset):\n",
"class TextDataset(Dataset):\n",

# =========================
# FRAMEWORKS (30)
# =========================
"app = Flask(__name__)\n",
"app = FastAPI()\n",
"router = APIRouter()\n",
"@app.route('/')\n",
"@app.get('/')\n",
"@app.post('/predict')\n",
"@torch.no_grad()\n",
"@dataclass\n",
"with torch.no_grad():\n",
"with open('data.txt') as f:\n",
"df = pd.read_csv(",
"plt.figure(",
"model = nn.Sequential(",
"optimizer = optim.Adam(",
"criterion = nn.CrossEntropyLoss(",
"train_loader = DataLoader(",
"test_loader = DataLoader(",
"torch.save(",
"torch.load(",
"np.array(",
"np.random.seed(",
"pd.DataFrame(",
"json.loads(",
"json.dumps(",
"requests.get(",
"requests.post(",
"logging.basicConfig(",
"async def fetch(url):\n",
"if __name__ == '__main__':\n",
"main()\n",

# =========================
# EXTRA (30)
# =========================
"SELECT * FROM users",
"CREATE TABLE users(",
"pip install ",
"docker run ",
"git clone ",
"FROM python:3.11",
"requirements = [",
"setup.py",
"pyproject.toml",
"README.md",
"config = {",
"device = torch.device(",
"loss.backward()",
"optimizer.step()",
"scheduler.step()",
"model.eval()",
"model.train()",
"torch.cuda.is_available()",
"np.mean(",
"np.std(",
"pd.merge(",
"plt.plot(",
"fig, ax = plt.subplots(",
"logging.getLogger(",
"Path(",
"os.path.join(",
"argparse.ArgumentParser(",
"asyncio.run(",
"ThreadPoolExecutor(",
"ProcessPoolExecutor("
]
for prompt in test_prompts:
    ids = torch.tensor([tokenizer.encode(prompt)], device=device)
    print("="*60)
    print("PROMPT:", prompt)
    for name, model in [("BASE", base_model), ("DISTILLED", distilled_model)]:
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=80, temperature=0.7, top_k=50,
                                  eos_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0].tolist())
        print(f"\n--- {name} ---\n{text}")

Streaming output truncated to the last 5000 lines.
--- BASE ---
import itertools
import glob
import os
import shutil
import subprocess
import sys

from django.contrib.staticfiles.storage import staticfiles_storage

from . import serve
from .config import get_config


class StaticFilesStorage(object):
    def __init__(self, storage_class):
        self.storage_class = storage_

--- DISTILLED ---
import itertools
import json
import re
import sys

import requests

from flask import Flask, request
from flask_tasks import Task
from flask_tasks import TaskSet
from flask_tasks import TaskQueue
from flask_tasks import Task

from bs4 import BeautifulSoup

from django.conf import settings
PROMPT: import collections


--- BASE ---
import collections

import functools
import numpy as np
from pandas.core.dtypes.common import is_sequence_like
from pandas.core.dtypes.common import is_sequence_like
from pandas.core.dtypes.common import is_sequence_like_no_dtype
from pandas.core.dtypes.common import is